In [1]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append(".")
print("Ready")

Ready


In [2]:
%%writefile tools/evidence.py
from typing import Dict, List
from urllib.parse import urlparse

MIN_INDEPENDENT_SOURCES = 2

# Fine to read, but not counted as independent corroboration.
LOW_QUALITY_DOMAINS = {
    "youtube.com", "youtu.be", "facebook.com", "tiktok.com", "pinterest.com",
    "instagram.com", "x.com", "twitter.com", "quora.com",
}

_TWO_PART_SUFFIXES = {"co.uk", "com.au", "co.in", "com.cn", "co.jp", "com.br", "co.kr"}


def _registrable(host: str) -> str:
    host = host.lower().split(":")[0]
    if host.startswith("www."):
        host = host[4:]
    parts = host.split(".")
    if len(parts) >= 3 and ".".join(parts[-2:]) in _TWO_PART_SUFFIXES:
        return ".".join(parts[-3:])
    if len(parts) >= 2:
        return ".".join(parts[-2:])
    return host


def source_domain(source: Dict) -> str:
    url = source.get("url", "") or ""
    host = urlparse(url).hostname or ""
    # Gemini grounding links redirect through a Google domain; the real site is in the title.
    if host.endswith("vertexaisearch.cloud.google.com"):
        title = (source.get("title") or "").strip().lower()
        if "." in title and " " not in title:
            return _registrable(title)
    return _registrable(host) if host else ""


def independent_domains(sources: List[Dict]) -> set:
    domains = {source_domain(s) for s in sources}
    domains.discard("")
    return domains - LOW_QUALITY_DOMAINS


def independent_source_count(sources: List[Dict]) -> int:
    return len(independent_domains(sources))

Writing tools/evidence.py


In [3]:
%%writefile tools/llm_clients.py
import os
import time
import threading
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from google import genai
from google.genai import types
from google.genai import errors as genai_errors
from tavily import TavilyClient
from ddgs import DDGS

load_dotenv()

GROQ_KEY = os.getenv("GROQ_API_KEY")
GEMINI_KEY = os.getenv("GEMINI_API_KEY")
TAVILY_KEY = os.getenv("TAVILY_API_KEY")
DEMO_MODE = os.getenv("DEMO_MODE", "false").lower() == "true"

assert GROQ_KEY, "Missing GROQ_API_KEY"
assert GEMINI_KEY, "Missing GEMINI_API_KEY"
assert TAVILY_KEY, "Missing TAVILY_API_KEY"

reasoning_llm = ChatGroq(model="qwen/qwen3.8-27b", temperature=0)
gemini_client = genai.Client(api_key=GEMINI_KEY)
tavily_client = TavilyClient(api_key=TAVILY_KEY)

MAX_CONCURRENT_SEARCHES = 1 if DEMO_MODE else 2
_search_semaphore = threading.Semaphore(MAX_CONCURRENT_SEARCHES)


def is_rate_limit_error(e: Exception) -> bool:
    msg = str(e).lower()
    return "429" in msg or "rate limit" in msg or "resource_exhausted" in msg


def call_llm(prompt: str, max_retries: int = 2) -> str:
    last_error = None
    for attempt in range(max_retries + 1):
        try:
            return reasoning_llm.invoke(prompt).content
        except Exception as e:
            last_error = e
            if is_rate_limit_error(e) and attempt < max_retries:
                time.sleep(2 ** attempt)
                continue
            raise
    raise last_error


def _mark_snippets(snippets) -> str:
    """Tags each snippet [S1], [S2]... so extraction can say which source states a claim (Day 23)."""
    return " ".join(f"[S{i}] {text[:300]}" for i, text in enumerate(snippets, start=1))


def _search_duckduckgo(query: str) -> dict:
    results = list(DDGS().text(query, max_results=3))
    sources = [
        {"id": i, "title": r.get("title", ""), "url": r.get("href") or r.get("url", "")}
        for i, r in enumerate(results, start=1)
    ]
    answer = _mark_snippets([r.get("body", "") for r in results])
    return {"answer": answer, "sources": sources, "provider": "duckduckgo"}


def research_with_fallback(query: str) -> dict:
    """Gemini -> Tavily -> DuckDuckGo, concurrency-limited by a semaphore."""
    with _search_semaphore:
        try:
            resp = gemini_client.models.generate_content(
                model="gemini-3.6-flash",
                contents=query,
                config=types.GenerateContentConfig(tools=[types.Tool(google_search=types.GoogleSearch())])
            )
            sources = []
            if resp.candidates[0].grounding_metadata:
                sources = [
                    {"id": i, "title": c.web.title, "url": c.web.uri}
                    for i, c in enumerate(resp.candidates[0].grounding_metadata.grounding_chunks, start=1)
                ]
            # Gemini's answer is one synthesized text, so it has no per-snippet tags.
            return {"answer": resp.text, "sources": sources, "provider": "gemini"}
        except (genai_errors.ClientError, genai_errors.ServerError) as e:
            print(f"Gemini unavailable ({type(e).__name__}), falling back to Tavily...")

        try:
            results = tavily_client.search(query=query, max_results=3)["results"]
            sources = [{"id": i, "title": r["title"], "url": r["url"]} for i, r in enumerate(results, start=1)]
            answer = _mark_snippets([r.get("content", "") for r in results])
            return {"answer": answer, "sources": sources, "provider": "tavily"}
        except Exception as e:
            print(f" Tavily unavailable ({type(e).__name__}), falling back to DuckDuckGo...")

        try:
            return _search_duckduckgo(query)
        except Exception as e:
            print(f" DuckDuckGo also unavailable ({type(e).__name__})")
            return {"answer": "", "sources": [], "provider": "none"}

Overwriting tools/llm_clients.py


In [4]:
%%writefile agents/data_extraction_agent.py
import json
import re
from typing import Dict, List

from tools.llm_clients import call_llm
from tools.logger import log_agent_event

AGENT_NAME = "data_extraction_agent"

# Day 23: ask the model which snippet(s) support each claim. This costs a few extra output
# tokens per fact. If Groq's per-minute budget gets tight, set it to False: claims then
# inherit all of the search result's sources and are flagged attribution="shared".
ATTRIBUTE_CLAIMS = True


def _build_extraction_prompt(sub_question: str, answer: str, attribute: bool = True) -> str:
    if attribute:
        output_rules = (
            "- The text is split into snippets tagged [S1], [S2], and so on. For each fact, list the\n"
            "  numbers of the snippets that actually state it. Do not guess: if only one snippet\n"
            "  states a fact, list only that one.\n"
            '- Respond with ONLY a JSON array of objects, nothing else: [{"claim": "...", "sources": [1, 3]}]\n'
            "- If there are no clear factual claims, return []."
        )
    else:
        output_rules = (
            "- Respond with ONLY a JSON array of strings, nothing else.\n"
            "- If there are no clear factual claims, return []."
        )
    return f"""Extract discrete, atomic factual claims from the text below, related to the
question: "{sub_question}"

Rules:
- Each fact must be a single, self-contained, checkable statement.
- Do NOT include opinions, vague generalities, or marketing language.
{output_rules}

Text:
\"\"\"{answer}\"\"\"
"""


def _parse_json_facts(raw_text: str) -> List[Dict]:
    """Returns [{"claim": str, "source_ids": [int, ...]}]. Accepts the object format and
    the older plain-string format."""
    cleaned = re.sub(r"^```(json)?", "", raw_text.strip()).strip()
    cleaned = re.sub(r"```$", "", cleaned).strip()
    try:
        parsed = json.loads(cleaned)
    except json.JSONDecodeError:
        return []
    if not isinstance(parsed, list):
        return []

    facts = []
    for item in parsed:
        if isinstance(item, str):
            claim, ids = item.strip(), []
        elif isinstance(item, dict):
            claim = str(item.get("claim", "")).strip()
            raw_ids = item.get("sources", [])
            ids = [i for i in raw_ids if isinstance(i, int)] if isinstance(raw_ids, list) else []
        else:
            continue
        if claim:
            facts.append({"claim": claim, "source_ids": ids})
    return facts


def _resolve_sources(source_ids: List[int], all_sources: List[Dict]) -> List[Dict]:
    by_id = {s.get("id"): s for s in all_sources if s.get("id") is not None}
    return [by_id[i] for i in dict.fromkeys(source_ids) if i in by_id]


def extract_facts_from_result(search_result: Dict) -> List[Dict]:
    sub_question = search_result.get("sub_question", "")
    answer = search_result.get("answer", "")
    sources = search_result.get("sources", [])

    if not answer.strip():
        return []

    attribute = ATTRIBUTE_CLAIMS and "[S1]" in answer

    try:
        raw_output = call_llm(_build_extraction_prompt(sub_question, answer, attribute))
        raw_facts = _parse_json_facts(raw_output)
    except Exception as e:
        log_agent_event(AGENT_NAME, "extraction_failed", sub_question=sub_question,
                        error=f"{type(e).__name__}: {e}")
        return []

    facts = []
    for item in raw_facts:
        if not attribute:
            claim_sources, attribution = sources, "shared"
        else:
            cited = _resolve_sources(item["source_ids"], sources)
            if cited:
                claim_sources, attribution = cited, "per_claim"
            else:
                # Tagged text but no valid citation: fail toward UNVERIFIED, not toward trusted.
                claim_sources, attribution = [], "unattributed"
        facts.append({"claim": item["claim"], "sub_question": sub_question,
                      "sources": claim_sources, "attribution": attribution})

    log_agent_event(AGENT_NAME, "extraction_done", sub_question=sub_question, facts_found=len(facts))
    return facts


def data_extraction_node(state: Dict) -> Dict:
    search_results = state.get("search_results", [])
    if not search_results:
        return {
            "errors": [{"agent": AGENT_NAME, "error": "no search_results to extract from"}],
            "model_usage_log": [{"agent": AGENT_NAME, "status": "skipped_no_input"}],
        }

    all_facts: List[Dict] = []
    errors: List[Dict] = []
    for result in search_results:
        try:
            all_facts.extend(extract_facts_from_result(result))
        except Exception as e:
            errors.append({"agent": AGENT_NAME, "sub_question": result.get("sub_question", "unknown"),
                           "error": f"{type(e).__name__}: {e}"})

    return {
        "extracted_facts": all_facts,
        "errors": errors,
        "model_usage_log": [{"agent": AGENT_NAME, "provider": "groq", "calls": len(search_results)}],
    }

Overwriting agents/data_extraction_agent.py


In [5]:
%%writefile agents/fact_checking_agent.py
import json
import re
from typing import Dict, List

from tools.llm_clients import call_llm
from tools.logger import log_agent_event
from tools.evidence import independent_source_count, MIN_INDEPENDENT_SOURCES

AGENT_NAME = "fact_checking_agent"
MAX_FACTS_PER_BATCH = 15


def _build_fact_check_prompt(facts: List[Dict]) -> str:
    numbered = "\n".join(
        f'{i}. "{f["claim"]}" (independent sources: {independent_source_count(f.get("sources", []))})'
        for i, f in enumerate(facts)
    )
    return f"""You are fact-checking a set of claims gathered during research. For EACH
numbered claim below, classify it as exactly one of:

- CONFIRMED: at least {MIN_INDEPENDENT_SOURCES} independent sources (see the count) and nothing in this list contradicts it
- DISPUTED: contradicts another claim in this list
- UNVERIFIED: fewer than {MIN_INDEPENDENT_SOURCES} independent sources, and no contradiction

Claims:
{numbered}

Respond with ONLY a JSON array: [{{"index": 0, "status": "CONFIRMED", "reasoning": "one short sentence"}}, ...]
"""


def _parse_fact_check_response(raw_text: str, expected_count: int) -> List[Dict]:
    cleaned = re.sub(r"^```(json)?", "", raw_text.strip()).strip()
    cleaned = re.sub(r"```$", "", cleaned).strip()
    try:
        parsed = json.loads(cleaned)
        if not isinstance(parsed, list):
            return []
        results = {}
        for item in parsed:
            idx = item.get("index")
            if isinstance(idx, int) and 0 <= idx < expected_count:
                status = item.get("status", "UNVERIFIED").upper()
                if status not in ("CONFIRMED", "DISPUTED", "UNVERIFIED"):
                    status = "UNVERIFIED"
                results[idx] = {"status": status, "reasoning": item.get("reasoning", "")}
        return [results.get(i, {"status": "UNVERIFIED", "reasoning": "not returned by model"})
                for i in range(expected_count)]
    except json.JSONDecodeError:
        return []


def apply_evidence_gate(fact: Dict, verdict: Dict) -> Dict:
    """Deterministic check applied AFTER the LLM verdict: a claim cannot stay CONFIRMED
    unless it has enough independent source domains, whatever the model said."""
    count = independent_source_count(fact.get("sources", []))
    status, reasoning = verdict["status"], verdict["reasoning"]
    downgraded = False
    if status == "CONFIRMED" and count < MIN_INDEPENDENT_SOURCES:
        status = "UNVERIFIED"
        reasoning = f"[downgraded: {count} independent source(s), need {MIN_INDEPENDENT_SOURCES}] {reasoning}"
        downgraded = True
    return {**fact, "status": status, "reasoning": reasoning,
            "independent_sources": count, "downgraded": downgraded}


def fact_check_batch(facts: List[Dict]) -> List[Dict]:
    if not facts:
        return []
    try:
        raw_output = call_llm(_build_fact_check_prompt(facts))
        verdicts = _parse_fact_check_response(raw_output, len(facts))
        if not verdicts:
            raise ValueError("fact-check response could not be parsed")
    except Exception as e:
        log_agent_event(AGENT_NAME, "batch_failed", error=f"{type(e).__name__}: {e}", batch_size=len(facts))
        verdicts = [{"status": "UNVERIFIED", "reasoning": "fact-check call failed; defaulted to unverified"}
                    for _ in facts]

    checked = [apply_evidence_gate(f, v) for f, v in zip(facts, verdicts)]

    counts: Dict[str, int] = {}
    for c in checked:
        counts[c["status"]] = counts.get(c["status"], 0) + 1
    log_agent_event(AGENT_NAME, "batch_done", downgraded=sum(1 for c in checked if c["downgraded"]), **counts)
    return checked


def fact_check_facts(facts: List[Dict]) -> List[Dict]:
    all_checked = []
    for i in range(0, len(facts), MAX_FACTS_PER_BATCH):
        all_checked.extend(fact_check_batch(facts[i:i + MAX_FACTS_PER_BATCH]))
    return all_checked


def fact_checking_node(state: Dict) -> Dict:
    extracted_facts = state.get("extracted_facts", [])
    if not extracted_facts:
        return {
            "errors": [{"agent": AGENT_NAME, "error": "no extracted_facts to check"}],
            "model_usage_log": [{"agent": AGENT_NAME, "status": "skipped_no_input"}],
        }

    checked = fact_check_facts(extracted_facts)
    return {
        "fact_check_results": checked,
        "model_usage_log": [{"agent": AGENT_NAME, "provider": "groq", "facts_checked": len(checked)}],
    }

Overwriting agents/fact_checking_agent.py


In [6]:
path = "agents/report_generator_agent.py"
with open(path, encoding="utf-8") as f:
    src = f.read()

anchor = '    disputed = [f for f in fact_check_results if f["status"] == "DISPUTED"]\n'
addition = '''    downgraded = [f for f in fact_check_results if f.get("downgraded")]
    if downgraded:
        lines.append(f"**Held back by the evidence gate (fewer than 2 independent sources):** {len(downgraded)}\\n")

'''
assert src.count(anchor) == 1, "anchor not found exactly once; paste me _build_confidence_appendix"
if "Held back by the evidence gate" not in src:
    src = src.replace(anchor, addition + anchor)
    with open(path, "w", encoding="utf-8") as f:
        f.write(src)
print(" report appendix patched")

 report appendix patched


In [7]:
%%writefile -a evaluate.py


def eval_confirmed_claims_have_independent_sources(result: Dict) -> Dict:
    """Day 23: every CONFIRMED claim must carry the minimum number of independent source domains."""
    from tools.evidence import independent_source_count, MIN_INDEPENDENT_SOURCES
    confirmed = [f for f in result.get("fact_check_results", []) if f.get("status") == "CONFIRMED"]
    weak = [f["claim"] for f in confirmed
            if independent_source_count(f.get("sources", [])) < MIN_INDEPENDENT_SOURCES]
    return {
        "name": "confirmed_claims_have_independent_sources",
        "passed": not weak,
        "detail": f"{len(confirmed)} confirmed, {len(weak)} below the {MIN_INDEPENDENT_SOURCES}-source bar",
    }


def eval_disputes_surfaced_in_narrative(result: Dict) -> Dict:
    """Replaces eval_disputes_are_surfaced, which could never fail because the appendix
    always contains 'Disputed'. This only looks at the narrative above the appendix."""
    disputed = [f for f in result.get("fact_check_results", []) if f.get("status") == "DISPUTED"]
    if not disputed:
        return {"name": "disputes_surfaced_in_narrative", "passed": True,
                "detail": "no disputed claims this run (n/a)"}
    narrative = (result.get("final_report") or "").split("## Confidence Appendix")[0].lower()
    ok = any(w in narrative for w in ["dispute", "disagree", "conflict", "contradic", "contested", "debate"])
    return {"name": "disputes_surfaced_in_narrative", "passed": ok,
            "detail": f"{len(disputed)} disputed claim(s); dispute language in narrative: {ok}"}


ALL_CHECKS[:] = [c for c in ALL_CHECKS if c is not eval_disputes_are_surfaced] + [
    eval_disputes_surfaced_in_narrative,
    eval_confirmed_claims_have_independent_sources,
]

Appending to evaluate.py


In [8]:
%%writefile tests/test_evidence.py
from tools.evidence import source_domain, independent_source_count


def test_domain_strips_www_and_subdomains():
    assert source_domain({"url": "https://www.nature.com/articles/x"}) == "nature.com"
    assert source_domain({"url": "https://news.bbc.co.uk/a"}) == "bbc.co.uk"


def test_same_domain_counts_once():
    sources = [{"url": "http://nature.com/1"}, {"url": "http://www.nature.com/2"}, {"url": "http://nature.com/3"}]
    assert independent_source_count(sources) == 1


def test_distinct_domains_counted():
    assert independent_source_count([{"url": "http://nature.com/1"}, {"url": "http://science.org/2"}]) == 2


def test_low_quality_domains_not_counted():
    sources = [{"url": "https://www.youtube.com/watch?v=abc"}, {"url": "http://nature.com/1"}]
    assert independent_source_count(sources) == 1


def test_gemini_redirect_uses_title_domain():
    sources = [
        {"url": "https://vertexaisearch.cloud.google.com/grounding-api-redirect/abc", "title": "nature.com"},
        {"url": "https://vertexaisearch.cloud.google.com/grounding-api-redirect/def", "title": "science.org"},
    ]
    assert independent_source_count(sources) == 2


def test_empty_and_malformed_sources():
    assert independent_source_count([]) == 0
    assert independent_source_count([{"url": ""}, {}]) == 0

Writing tests/test_evidence.py


In [9]:
%%writefile tests/test_attribution.py
import agents.data_extraction_agent as dea

SOURCES = [
    {"id": 1, "title": "A", "url": "http://a.com/1"},
    {"id": 2, "title": "B", "url": "http://b.com/2"},
]
MARKED = "[S1] first snippet text. [S2] second snippet text."


def _extract(monkeypatch, llm_output, answer=MARKED):
    monkeypatch.setattr(dea, "call_llm", lambda p: llm_output)
    return dea.extract_facts_from_result({"sub_question": "q", "answer": answer, "sources": SOURCES})


def test_per_claim_attribution_keeps_only_cited_sources(monkeypatch):
    facts = _extract(monkeypatch, '[{"claim": "X", "sources": [2]}]')
    assert facts[0]["sources"] == [SOURCES[1]]
    assert facts[0]["attribution"] == "per_claim"


def test_unknown_source_id_is_unattributed(monkeypatch):
    facts = _extract(monkeypatch, '[{"claim": "X", "sources": [9]}]')
    assert facts[0]["sources"] == []
    assert facts[0]["attribution"] == "unattributed"


def test_plain_string_output_on_tagged_text_is_unattributed(monkeypatch):
    facts = _extract(monkeypatch, '["X"]')
    assert facts[0]["sources"] == []
    assert facts[0]["attribution"] == "unattributed"


def test_untagged_text_shares_all_sources(monkeypatch):
    facts = _extract(monkeypatch, '["X"]', answer="plain synthesized answer with no tags")
    assert facts[0]["sources"] == SOURCES
    assert facts[0]["attribution"] == "shared"


def test_attribution_flag_off_shares_all_sources(monkeypatch):
    monkeypatch.setattr(dea, "ATTRIBUTE_CLAIMS", False)
    facts = _extract(monkeypatch, '["X"]')
    assert facts[0]["sources"] == SOURCES
    assert facts[0]["attribution"] == "shared"

Writing tests/test_attribution.py


In [10]:
%%writefile tests/test_fact_checking_agent.py
import json
import agents.fact_checking_agent as fca

TWO_INDEPENDENT = [{"url": "http://a.com/1"}, {"url": "http://b.com/2"}]
ONE_SOURCE = [{"url": "http://a.com/1"}]
SAME_DOMAIN = [{"url": "http://a.com/1"}, {"url": "http://a.com/2"}]


def _verdict_llm(status):
    return lambda p: json.dumps([{"index": 0, "status": status, "reasoning": "model said so"}])


def test_batch_parses_valid_response(monkeypatch):
    monkeypatch.setattr(fca, "call_llm", lambda p: json.dumps([
        {"index": 0, "status": "CONFIRMED", "reasoning": "two sources agree"},
        {"index": 1, "status": "DISPUTED", "reasoning": "contradicts claim 0"},
    ]))
    facts = [{"claim": "A", "sources": TWO_INDEPENDENT}, {"claim": "B", "sources": TWO_INDEPENDENT}]
    checked = fca.fact_check_batch(facts)
    assert checked[0]["status"] == "CONFIRMED"
    assert checked[1]["status"] == "DISPUTED"


def test_batch_defaults_to_unverified_on_llm_failure(monkeypatch):
    def boom(p):
        raise RuntimeError("down")
    monkeypatch.setattr(fca, "call_llm", boom)
    checked = fca.fact_check_batch([{"claim": "A", "sources": TWO_INDEPENDENT}])
    assert checked[0]["status"] == "UNVERIFIED"


def test_batch_defaults_to_unverified_on_malformed_json(monkeypatch):
    monkeypatch.setattr(fca, "call_llm", lambda p: "not json")
    checked = fca.fact_check_batch([{"claim": "A", "sources": TWO_INDEPENDENT}])
    assert checked[0]["status"] == "UNVERIFIED"


def test_batch_fills_missing_index_as_unverified(monkeypatch):
    monkeypatch.setattr(fca, "call_llm", lambda p: json.dumps([
        {"index": 0, "status": "CONFIRMED", "reasoning": "ok"}
    ]))
    facts = [{"claim": "A", "sources": TWO_INDEPENDENT}, {"claim": "B", "sources": TWO_INDEPENDENT}]
    checked = fca.fact_check_batch(facts)
    assert checked[0]["status"] == "CONFIRMED"
    assert checked[1]["status"] == "UNVERIFIED"


def test_empty_facts_returns_empty(monkeypatch):
    assert fca.fact_check_facts([]) == []


def test_chunking_splits_large_batches(monkeypatch):
    calls = {"count": 0}
    def fake_call(p):
        calls["count"] += 1
        return json.dumps([{"index": i, "status": "UNVERIFIED", "reasoning": "x"}
                           for i in range(fca.MAX_FACTS_PER_BATCH)])
    monkeypatch.setattr(fca, "call_llm", fake_call)
    facts = [{"claim": f"fact {i}", "sources": []} for i in range(fca.MAX_FACTS_PER_BATCH + 5)]
    checked = fca.fact_check_facts(facts)
    assert len(checked) == fca.MAX_FACTS_PER_BATCH + 5
    assert calls["count"] == 2


def test_node_empty_extracted_facts(monkeypatch):
    out = fca.fact_checking_node({"extracted_facts": []})
    assert out["errors"] and "fact_check_results" not in out


def test_gate_downgrades_confirmed_with_one_independent_source(monkeypatch):
    monkeypatch.setattr(fca, "call_llm", _verdict_llm("CONFIRMED"))
    checked = fca.fact_check_batch([{"claim": "A", "sources": ONE_SOURCE}])
    assert checked[0]["status"] == "UNVERIFIED"
    assert checked[0]["downgraded"] is True
    assert checked[0]["independent_sources"] == 1


def test_gate_keeps_confirmed_with_two_independent_sources(monkeypatch):
    monkeypatch.setattr(fca, "call_llm", _verdict_llm("CONFIRMED"))
    checked = fca.fact_check_batch([{"claim": "A", "sources": TWO_INDEPENDENT}])
    assert checked[0]["status"] == "CONFIRMED"
    assert checked[0]["downgraded"] is False
    assert checked[0]["independent_sources"] == 2


def test_gate_does_not_touch_disputed(monkeypatch):
    monkeypatch.setattr(fca, "call_llm", _verdict_llm("DISPUTED"))
    checked = fca.fact_check_batch([{"claim": "A", "sources": ONE_SOURCE}])
    assert checked[0]["status"] == "DISPUTED"
    assert checked[0]["downgraded"] is False


def test_gate_same_domain_sources_are_not_independent(monkeypatch):
    monkeypatch.setattr(fca, "call_llm", _verdict_llm("CONFIRMED"))
    checked = fca.fact_check_batch([{"claim": "A", "sources": SAME_DOMAIN}])
    assert checked[0]["status"] == "UNVERIFIED"

Overwriting tests/test_fact_checking_agent.py


In [11]:
%%writefile -a tests/test_report_generator_agent.py


def test_appendix_reports_downgraded_count():
    held_back = [
        {"claim": "A", "status": "UNVERIFIED", "reasoning": "[downgraded]", "sources": [], "downgraded": True},
        {"claim": "B", "status": "CONFIRMED", "reasoning": "", "sources": [], "downgraded": False},
    ]
    assert "Held back by the evidence gate" in rga._build_confidence_appendix(held_back)

    clean = [{"claim": "B", "status": "CONFIRMED", "reasoning": "", "sources": [], "downgraded": False}]
    assert "evidence gate" not in rga._build_confidence_appendix(clean)

Appending to tests/test_report_generator_agent.py


In [ ]:
from collections import Counter
from graph.build_graph import build_graph
from evaluate import evaluate_run, print_eval_report

app = build_graph()
q = "Compare the battery recycling strategies of Tesla, Redwood Materials, and CATL"
result = app.invoke({
    "query": q, "plan": [], "next_agent": "", "iteration_count": 0, "needs_more_research": False,
    "search_results": [], "extracted_facts": [], "fact_check_results": [], "summaries": [],
    "errors": [], "final_report": None, "model_usage_log": [],
})

facts = result["fact_check_results"]
print("Attribution modes:", dict(Counter(f.get("attribution", "n/a") for f in facts)))
print("Status after gate:", dict(Counter(f["status"] for f in facts)))
print("Held back by gate:", sum(1 for f in facts if f.get("downgraded")))
print("Independent source counts:", dict(Counter(f.get("independent_sources") for f in facts)))
print_eval_report(q, evaluate_run(result))